<a href="https://colab.research.google.com/github/Shwoouu/AAI2025/blob/2026fall/Group_Exercise_Agentic_AI_in_Customer_Service_and_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai langgraph langchain-core

import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
except ImportError:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    raise ValueError(
        "GEMINI_API_KEY not found! Please set it in Colab Secrets or as an environment variable (`export GEMINI_API_KEY='your_key'`)."
    )


from typing import Literal, TypedDict
from google import genai
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

client = genai.Client()

class AgentState(TypedDict):
    messages: list[BaseMessage]
    next_agent: str

def router_node(state: AgentState):
    """Analyzes customer intent and directs query to the correct department."""
    messages = state["messages"]
    latest_message = messages[-1].content

    prompt = f"""
    You are a customer service routing assistant. Analyze the user's message and categorize it into exactly one of these three departments:
    - 'order_status': For tracking, shipping, or delivery questions.
    - 'refund_policy': For returns, refunds, exchanges, or money-back guarantees.
    - 'product_suggestion': For recommendations, product features, or shopping assistance.

    User Message: "{latest_message}"

    Respond with ONLY the department name: order_status, refund_policy, or product_suggestion.
    """

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt
    )

    category = response.text.strip().lower()
    if category not in ["order_status", "refund_policy", "product_suggestion"]:
        category = "product_suggestion"

    return {"next_agent": category}

def order_status_agent(state: AgentState):
    messages = state["messages"]
    history = "\n".join([f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}" for m in messages[-4:]])

    prompt = f"""
    You are a helpful Order Status and Logistics customer service agent.
    Be polite, concise, and ask for an order tracking number if it's missing.

    Conversation History:
    {history}

    Provide your response as the assistant:
    """
    response = client.models.generate_content(model='gemini-3.5-flash-lite', contents=prompt)
    return {"messages": messages + [AIMessage(content=response.text)]}

def refund_policy_agent(state: AgentState):
    messages = state["messages"]
    history = "\n".join([f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}" for m in messages[-4:]])

    prompt = f"""
    You are a Refund and Returns Policy specialist.
    Our policy is a 30-day money-back guarantee on all unused items in original packaging.
    Be clear, empathetic, and informative.

    Conversation History:
    {history}

    Provide your response as the assistant:
    """
    response = client.models.generate_content(model='gemini-3.5-flash-lite', contents=prompt)
    return {"messages": messages + [AIMessage(content=response.text)]}

def product_suggestion_agent(state: AgentState):
    messages = state["messages"]
    history = "\n".join([f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}" for m in messages[-4:]])

    prompt = f"""
    You are an expert Sales and Product Recommendation assistant.
    Help customers find the best products based on their needs, budget, or preferences.

    Conversation History:
    {history}

    Provide your response as the assistant:
    """
    response = client.models.generate_content(model='gemini-3.5-flash-lite', contents=prompt)
    return {"messages": messages + [AIMessage(content=response.text)]}

workflow = StateGraph(AgentState)

workflow.add_node("router", router_node)
workflow.add_node("order_status", order_status_agent)
workflow.add_node("refund_policy", refund_policy_agent)
workflow.add_node("product_suggestion", product_suggestion_agent)

workflow.add_edge(START, "router")
workflow.add_conditional_edges(
    "router",
    lambda state: state["next_agent"],
    {
        "order_status": "order_status",
        "refund_policy": "refund_policy",
        "product_suggestion": "product_suggestion"
    }
)
workflow.add_edge("order_status", END)
workflow.add_edge("refund_policy", END)
workflow.add_edge("product_suggestion", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

if __name__ == "__main__":

    config = {"configurable": {"thread_id": "customer_session_colab_1"}}

    test_inputs = [
        "Hi, can you recommend a good laptop for video editing under $1500?",
        "What is your return policy if I end up not liking it?",
        "Great, how can I track my shipment once I buy it?"
    ]

    print("--- Starting Multi-Agent Conversation Test ---")
    for i, user_input in enumerate(test_inputs, 1):
        print(f"\n[Turn {i}]")
        print(f"User: {user_input}")

        output = app.invoke({"messages": [HumanMessage(content=user_input)]}, config)

        routed_to = output.get("next_agent", "unknown")
        latest_response = output["messages"][-1].content.strip()

        print(f"-> [Routed to Agent: {routed_to}]")
        print(f"Assistant: {latest_response}")

--- Starting Multi-Agent Conversation Test ---

[Turn 1]
User: Hi, can you recommend a good laptop for video editing under $1500?
-> [Routed to Agent: product_suggestion]
Assistant: Hi there! I'd be happy to help you find a powerful laptop for video editing within your $1500 budget. 

For video editing, we want to look for a machine with a strong multi-core processor (CPU), at least 16GB of RAM (ideally 32GB if possible), a dedicated graphics card (GPU) to handle rendering, and a color-accurate display.

Here are three top recommendations that fit your budget and needs:

1. **ASUS ROG Zephyrus G14 (2023 / On Sale)**
   * **Why it’s great:** It packs an AMD Ryzen 9 processor and an NVIDIA RTX 4060 or 4070 graphics card into a sleek, highly portable 14-inch chassis. It has fantastic rendering performance and a gorgeous screen for color grading. 
   * **Price:** Often found on sale right around $1,300–$1,499.

2. **Apple MacBook Pro 14" (M2 Pro or M3, Refurbished or Base Model)**
   * **W